## The Dockerfile — `FROM` & `RUN`

A **Dockerfile** is a plain-text recipe for building an image, conventionally named `Dockerfile` (no extension) at your project root. The simplest possible one:

```dockerfile
FROM alpine:3.20
CMD ["echo", "hello from the image"]
```

```bash
docker build -t my-hello .    # read the Dockerfile in ., build, tag my-hello:latest
docker run --rm my-hello
```

**How a build executes.** Each instruction either *adds a layer* or *modifies the config blob*:

- **Filesystem-changing** (`RUN`, `COPY`, `ADD`) — run in a temporary container on the previous layer, then commit the result as a new layer.
- **Metadata-only** (`CMD`, `ENTRYPOINT`, `ENV`, `WORKDIR`, `USER`, `EXPOSE`, `LABEL`, `ARG`) — change the JSON config, no new bytes.

**`FROM` — the base layer.** Every Dockerfile starts here; it picks the image everything else stacks on.

- **Official images** (`python`, `node`, `golang`) — maintained by Docker or the language teams; the default choice.
- **`-slim`** drops everything not needed for the runtime — smaller, fewer CVEs.
- **`-alpine`** uses Alpine (musl libc) — smallest, but test for binary-wheel compatibility.
- **`scratch`** — the literal empty image; base for statically-linked Go/Rust binaries.
- **Pin the tag** (`python:3.12.7-slim-bookworm`, or a digest) for repeatable builds.

**`RUN <command>`** runs a command in a temporary container and commits a new layer — how you install packages and build code. Two forms: **shell form** (`RUN apt-get install -y curl`, via `/bin/sh -c`, gives you `&&`, pipes, `$VAR`) and **exec form** (`RUN ["apt-get","install","-y","curl"]`, no shell). Why you see `&&` everywhere: each `RUN` is a layer, so chain related steps into one to keep the layer count down, the package index fresh, and the cache clean:

```dockerfile
RUN apt-get update \
 && apt-get install -y --no-install-recommends curl ca-certificates \
 && rm -rf /var/lib/apt/lists/*
```